In [1]:
import os
import pandas as pd
from urllib.request import urlopen
from datetime import datetime

if not os.path.exists('data'):
    os.makedirs('data')

def download_data():
    base_url = "https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={}&year1=1981&year2=2024&type=Mean"
    
    for i in range(1, 28):
        files = [f for f in os.listdir('data') if f.startswith(f'vhi_id_{i}_')]
        if files:
            print(f"Дані для області {i} вже завантажені.")
            continue
            
        try:
            url = base_url.format(i)
            with urlopen(url) as response:
                content = response.read()
                
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"data/vhi_id_{i}_{timestamp}.csv"
            
            with open(filename, 'wb') as f:
                f.write(content)
            print(f"Успішно завантажено область {i} -> {filename}")
        except Exception as e:
            print(f"Помилка при завантаженні області {i}: {e}")

download_data()

Успішно завантажено область 1 -> data/vhi_id_1_20260313_192845.csv
Успішно завантажено область 2 -> data/vhi_id_2_20260313_192846.csv
Успішно завантажено область 3 -> data/vhi_id_3_20260313_192847.csv
Успішно завантажено область 4 -> data/vhi_id_4_20260313_192848.csv
Успішно завантажено область 5 -> data/vhi_id_5_20260313_192849.csv
Успішно завантажено область 6 -> data/vhi_id_6_20260313_192850.csv
Успішно завантажено область 7 -> data/vhi_id_7_20260313_192851.csv
Успішно завантажено область 8 -> data/vhi_id_8_20260313_192853.csv
Успішно завантажено область 9 -> data/vhi_id_9_20260313_192854.csv
Успішно завантажено область 10 -> data/vhi_id_10_20260313_192855.csv
Успішно завантажено область 11 -> data/vhi_id_11_20260313_192856.csv
Успішно завантажено область 12 -> data/vhi_id_12_20260313_192857.csv
Успішно завантажено область 13 -> data/vhi_id_13_20260313_192858.csv
Успішно завантажено область 14 -> data/vhi_id_14_20260313_192859.csv
Успішно завантажено область 15 -> data/vhi_id_15_202

In [2]:
def clean_and_merge(folder_path):
    noaa_to_ua = {
        1: ("Черкаська", 22), 2: ("Чернігівська", 24), 3: ("Чернівецька", 23),
        4: ("Крим", 25), 5: ("Дніпропетровська", 3), 6: ("Донецька", 4),
        7: ("Івано-Франківська", 8), 8: ("Харківська", 19), 9: ("Херсонська", 20),
        10: ("Хмельницька", 21), 11: ("Київська", 9), 12: ("Київ", 26),
        13: ("Кіровоградська", 10), 14: ("Луганська", 11), 15: ("Львівська", 12),
        16: ("Миколаївська", 13), 17: ("Одеська", 14), 18: ("Полтавська", 15),
        19: ("Рівненська", 16), 20: ("Севастополь", 27), 21: ("Сумська", 17),
        22: ("Тернопільська", 18), 23: ("Вінницька", 1), 24: ("Волинська", 2),
        25: ("Закарпатська", 6), 26: ("Запорізька", 7), 27: ("Житомирська", 5)
    }

    all_data = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):
            orig_id = int(filename.split('_')[2])
            path = os.path.join(folder_path, filename)
            
            df = pd.read_csv(path, index_col=False, header=1, names=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'junk'])
            df = df.drop(columns=['junk'])
            df = df.dropna()
            
            df['Year'] = df['Year'].astype(str).str.replace('<tt><pre>', '', regex=False).astype(int)
            
            name, new_id = noaa_to_ua[orig_id]
            df['Province_Name'] = name
            df['Province_Index'] = new_id
            
            all_data.append(df)
            
    return pd.concat(all_data, ignore_index=True)

df = clean_and_merge('data')
df['Week'] = df['Week'].astype(int)
print("Дані успішно підготовлені. Перші 5 рядків:")
display(df.head())

Дані успішно підготовлені. Перші 5 рядків:


,Year,Week,SMN,SMT,VCI,TCI,VHI,Province_Name,Province_Index
0,1982,1,0.059,258.24,51.11,48.78,49.95,Хмельницька,21
1,1982,2,0.063,261.53,55.89,38.20,47.04,Хмельницька,21
2,1982,3,0.063,263.45,57.30,32.69,44.99,Хмельницька,21
3,1982,4,0.061,265.10,53.96,28.62,41.29,Хмельницька,21
4,1982,5,0.058,266.42,46.87,28.57,37.72,Хмельницька,21


In [3]:
def get_vhi_by_year(df, province_id, year):
    result = df[(df['Province_Index'] == province_id) & (df['Year'] == year)][['Week', 'VHI']]
    print(f"VHI для області {province_id} за {year} рік:")
    return result

def get_vhi_by_range(df, province_ids, start_year, end_year):
    result = df[(df['Province_Index'].isin(province_ids)) & 
                (df['Year'] >= start_year) & 
                (df['Year'] <= end_year)][['Province_Name', 'Year', 'Week', 'VHI']]
    return result

def get_vhi_stats(df, province_id, year_range):
    subset = df[(df['Province_Index'] == province_id) & 
                (df['Year'] >= year_range[0]) & (df['Year'] <= year_range[1])]
    
    stats = {
        'Min VHI': subset['VHI'].min(),
        'Max VHI': subset['VHI'].max(),
        'Mean VHI': subset['VHI'].mean(),
        'Median VHI': subset['VHI'].median()
    }
    return pd.Series(stats)

display(get_vhi_by_year(df, 1, 2020).head())
print("\nСтатистика для Вінницької області (ID 1) за 2010-2020:")
display(get_vhi_stats(df, 1, (2010, 2020)))

VHI для області 1 за 2020 рік:


,Week,VHI
33280,1,39.29
33281,2,39.08
33282,3,38.94
33283,4,39.47
33284,5,40.11



Статистика для Вінницької області (ID 1) за 2010-2020:


Min VHI       31.600000
Max VHI       69.280000
Mean VHI      49.707902
Median VHI    49.515000
dtype: float64